In [ ]:
import torch
import torchvision.models as models
import torch.onnx
import os

# 1. 모델 준비 (예: 학습된 ResNet18)
# 실제로는 이전에 학습한 가중치를 로드하거나 학습 코드가 있어야 합니다.
model_name = "resnet18"
model = models.resnet18(pretrained=True)
model.eval()

# 저장 경로 설정 (파일명 규칙 준수)
mission_prefix = f"mission_16_{model_name}"

# -------------------------------------------------------
# 타입 A: 기본 .pth 저장
# -------------------------------------------------------
torch.save(model.state_dict(), f"{mission_prefix}.pth")
print(f"1. 기본 모델 저장 완료: {mission_prefix}.pth")

# -------------------------------------------------------
# 타입 B: 양자화(Quantization)된 .pth 저장
# -------------------------------------------------------
# (참고) Dynamic Quantization은 LSTM/Linear에 효과적이며,
# CNN(ResNet 등)은 Static Quantization이 필요하지만,
# 실습 목적상 가장 간단한 Dynamic 방식을 예시로 듭니다.
# (에러가 난다면 qconfig를 사용하는 정적 양자화 방식을 써야 할 수 있습니다.)
import torch.quantization

# CPU 백엔드 설정 (양자화는 보통 CPU 추론 최적화용)
model_q = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},  # 양자화할 레이어 (Linear 등)
    dtype=torch.qint8
)
torch.save(model_q.state_dict(), f"{mission_prefix}_quantized.pth")
print(f"2. 양자화 모델 저장 완료: {mission_prefix}_quantized.pth")

# -------------------------------------------------------
# 타입 C: .onnx 저장
# -------------------------------------------------------
# ONNX 변환을 위한 더미 데이터 생성 (Input Shape에 맞춰야 함)
# 예: (Batch_size, Channel, Height, Width) -> (1, 3, 224, 224)
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,                      # 실행될 모델
    dummy_input,                # 모델 입력값 (차원 확인용)
    f"{mission_prefix}.onnx",   # 저장될 파일 경로
    export_params=True,         # 모델 파일 안에 학습된 가중치 포함 여부
    opset_version=11,           # ONNX 버전 (보통 11~13 권장)
    do_constant_folding=True,   # 최적화 여부
    input_names=['input'],      # 입력 노드 이름
    output_names=['output'],    # 출력 노드 이름
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}} # 가변 배치 사이즈 허용
)
print(f"3. ONNX 모델 저장 완료: {mission_prefix}.onnx")

# -------------------------------------------------------
# [보고서용] 용량 확인 코드
# -------------------------------------------------------
def get_file_size(file_path):
    size = os.path.getsize(file_path)
    return f"{size / 1024 / 1024:.2f} MB"

print("-" * 30)
print(f"기본 모델 용량: {get_file_size(f'{mission_prefix}.pth')}")
print(f"양자화 모델 용량: {get_file_size(f'{mission_prefix}_quantized.pth')}")
print(f"ONNX 모델 용량: {get_file_size(f'{mission_prefix}.onnx')}")
print("-" * 30)